# Capstone 19: Air Quality Level Classification
## Comprehensive Exploratory Data Analysis (EDA) on CPCB Indian Air Quality Data
**Author:** Antigravity Data Science & ML Engineering Team  
**Dataset:** Central Pollution Control Board (CPCB) India Monitoring Network (2015 - 2020)  
**Objective:** Characterize air pollutants, investigate seasonal/spatial patterns, evaluate class imbalances across CPCB NAQI categories, and establish clean preprocessing requirements for multi-class classification.


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure visual aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10
%matplotlib inline

print('All visualization and analytical packages imported successfully.')


## 1. Dataset Ingestion & Schema Inspection
We load data/city_day.csv containing daily air quality measurements recorded by the Central Pollution Control Board (CPCB), Ministry of Environment, Forest and Climate Change, Government of India.


In [ ]:
data_path = os.path.join('..', 'data', 'city_day.csv')
if not os.path.exists(data_path):
    data_path = 'data/city_day.csv'

df = pd.read_csv(data_path)
print(f'Total Records: {df.shape[0]:,}')
print(f'Total Columns: {df.shape[1]}')
display(df.head(5))


### Dataset Metadata & Variable Specifications
The primary chemical pollutants measured in micrograms per cubic meter ($\mu g/m^3$) — except CO which is measured in /m^3$ — are:
- **PM2.5 & PM10**: Fine and respirable suspended particulate matter
- **NO, NO2, NOx**: Nitrogen oxides originating from vehicular combustion and industrial thermal plants
- **NH3**: Ammonia emissions from agricultural runoff and livestock waste
- **CO**: Carbon monoxide produced by incomplete fossil fuel combustion
- **SO2**: Sulphur dioxide from coal-fired power stations and heavy fuel combustion
- **O3**: Tropospheric ground-level ozone formed by photochemical reactions
- **Benzene, Toluene, Xylene (BTX)**: Volatile organic compounds (VOCs)
- **AQI & AQI_Bucket**: The official ground-truth CPCB National Air Quality Index and its discrete severity category.


In [ ]:
df.info()
print('\nDescriptive Statistics for Pollutants:')
pollutant_cols = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene']
display(df[pollutant_cols].describe().T[['count', 'mean', 'std', 'min', '50%', 'max']])


## 2. Missing Value & Duplicate Analysis
Sensor maintenance outages, telemetry dropouts, and gradual sensor rollout in newly monitored cities introduce variable proportions of missingness across pollutants.


In [ ]:
# Check duplicates
dup_count = df.duplicated().sum()
print(f'Exact Duplicate Rows: {dup_count}')

# Missing value computation
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing_Count': missing, 'Percentage': missing_pct}).sort_values('Percentage', ascending=False)

plt.figure(figsize=(10, 5))
ax = sns.barplot(x=missing_df.index, y=missing_df['Percentage'], palette='mako')
plt.title('Missing Value Percentage by Column', fontsize=13, fontweight='bold', pad=10)
plt.ylabel('Missing Percentage (%)', fontsize=11)
plt.xticks(rotation=45, ha='right')
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f'{height:.1f}%', (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom', fontsize=8, xytext=(0, 2), textcoords='offset points')
plt.tight_layout()
plt.show()


### Interpretation:
1. Xylene, PM10, and NH3 experience the highest missingness (~61%, ~37%, and ~35% respectively) as automated Continuous Ambient Air Quality Monitoring Stations (CAAQMS) deployed specialized detectors for these compounds later in the national rollout.
2. CO, NO2, SO2, and PM2.5 have high completeness (< 15% missing), providing stable signals across all monitoring stations.
3. For target classification, 4,681 records lack the official CPCB target label (AQI_Bucket) due to insufficient concurrent pollutant sub-indices on those specific days. These are safely excluded from supervised training to prevent synthetic target bias.


## 3. Target Distribution & Class Imbalance Analysis
We examine the distribution of official CPCB AQI Buckets:
- **Good (0-50)**
- **Satisfactory (51-100)**
- **Moderate (101-200)**
- **Poor (201-300)**
- **Very Poor (301-400)**
- **Severe (401-500+)**


In [ ]:
labeled_df = df.dropna(subset=['AQI_Bucket']).copy()
cat_order = ['Good', 'Satisfactory', 'Moderate', 'Poor', 'Very Poor', 'Severe']
palette = ['#10B981', '#84CC16', '#F59E0B', '#F97316', '#EF4444', '#7F1D1D']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Count bar plot
sns.countplot(data=labeled_df, x='AQI_Bucket', order=cat_order, palette=palette, ax=ax1)
ax1.set_title('CPCB AQI Category Frequency', fontsize=12, fontweight='bold')
ax1.set_xlabel('AQI Category', fontsize=11)
ax1.set_ylabel('Number of Days', fontsize=11)
for p in ax1.patches:
    ax1.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2., p.get_height()),
                 ha='center', va='bottom', fontsize=9, xytext=(0, 2), textcoords='offset points')

# Pie chart proportions
counts = labeled_df['AQI_Bucket'].value_counts()[cat_order]
ax2.pie(counts, labels=cat_order, autopct='%1.1f%%', colors=palette, startangle=140, 
        wedgeprops=dict(width=0.4, edgecolor='w'))
ax2.set_title('Proportional Class Share', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()


### Interpretation:
- Moderate (35.5%) and Satisfactory (33.1%) represent over two-thirds of all recorded days in Indian cities.
- Extreme pollution categories — Severe (5.4%) and Good (5.4%) — represent distinct minority classes.
- Because of this class imbalance, evaluating models using simple overall Accuracy alone would be misleading (a majority-class classifier could achieve ~35% accuracy without learning true boundaries). **Macro F1-Score** and **Per-Class Recall** are vital evaluation metrics.


## 4. Pollutant Distributions & Outlier Checks
We inspect log-scale distributions and boxplots of key pollutants to identify typical ambient ranges and observe extreme episodic peaks.


In [ ]:
key_pollutants = ['PM2.5', 'PM10', 'NO2', 'CO', 'SO2', 'O3']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(key_pollutants):
    sns.histplot(labeled_df[col].dropna(), kde=True, ax=axes[i], color='#0D9488', bins=40)
    axes[i].set_title(f'{col} Distribution', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Concentration')
    axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()


### Interpretation:
- All chemical pollutants exhibit pronounced **positive (right) skewness**. Ambient air concentrations remain near baseline medians during fair weather but spike dramatically during winter temperature inversions, dust storms, and post-harvest crop residue burning events.
- Extreme concentrations of PM2.5 (> 500 $\mu g/m^3$) and PM10 (> 800 $\mu g/m^3$) represent genuine atmospheric crisis episodes (e.g. northern Indian winter smog) rather than sensor malfunction.


## 5. Correlation Analysis & Multi-Collinearity
We examine the Pearson correlation matrix to understand inter-pollutant relationships.


In [ ]:
plt.figure(figsize=(10, 8))
corr_matrix = labeled_df[pollutant_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', cbar_kws={'label': 'Pearson Correlation'},
            square=True, linewidths=0.5)
plt.title('Inter-Pollutant Correlation Heatmap', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()


### Interpretation:
1. **PM2.5 and PM10** exhibit strong positive correlation ( pprox 0.84$), consistent with atmospheric aerosol dynamics where fine particles are a primary constituent of coarse suspended dust.
2. **NO, NO2, and NOx** demonstrate strong mutual collinearity ( > 0.75$) due to shared vehicular combustion origin and rapid photochemical atmospheric inter-conversion.
3. **CO** shows moderate positive correlation with PM2.5 and NO2, indicating incomplete combustion from common urban transport and biomass sources.


## 6. Temporal & Seasonal Pollution Trends
Air quality in the Indian subcontinent varies heavily by season: winter months (November - January) bring cold air stagnation and smog, whereas monsoon months (July - September) offer atmospheric washout.


In [ ]:
labeled_df['Date'] = pd.to_datetime(labeled_df['Date'])
labeled_df['Year'] = labeled_df['Date'].dt.year
labeled_df['Month'] = labeled_df['Date'].dt.month

monthly_trend = labeled_df.groupby('Month')[['PM2.5', 'PM10', 'NO2', 'CO', 'AQI']].median()

fig, ax1 = plt.subplots(figsize=(12, 5))
color = 'tab:red'
ax1.set_xlabel('Month (1 = January, 12 = December)', fontsize=11)
ax1.set_ylabel('Particulate Matter Conc (ug/m3)', color=color, fontsize=11)
ax1.plot(monthly_trend.index, monthly_trend['PM2.5'], marker='o', linewidth=2.5, color='#EF4444', label='PM2.5 Median')
ax1.plot(monthly_trend.index, monthly_trend['PM10'], marker='s', linewidth=2, color='#F97316', label='PM10 Median')
ax1.tick_params(axis='y', labelcolor=color)
ax1.legend(loc='upper left')

ax2 = ax1.twinx()
color = 'tab:blue'
ax2.set_ylabel('AQI Value', color=color, fontsize=11)
ax2.plot(monthly_trend.index, monthly_trend['AQI'], marker='^', linewidth=2.5, color='#0284C7', linestyle='--', label='CPCB AQI Median')
ax2.tick_params(axis='y', labelcolor=color)
ax2.legend(loc='upper right')

plt.title('Seasonal Cycle: Monthly Median Particulate Matter & AQI', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()


### Interpretation:
- Air pollution exhibits a distinct U-shaped curve over the calendar year:
  - **Highest in November, December, and January**: Low temperatures and boundary layer atmospheric inversion trap particulate matter close to the ground.
  - **Lowest in July and August**: South-West Monsoon rains wash suspended particulate matter out of the troposphere (wet scavenging effect).


## 7. City-Wise Spatial Air Quality Comparison
Comparing average AQI across the most recorded Indian metropolitan areas.


In [ ]:
top_cities = labeled_df['City'].value_counts().head(10).index
city_df = labeled_df[labeled_df['City'].isin(top_cities)]

plt.figure(figsize=(12, 6))
order = city_df.groupby('City')['AQI'].median().sort_values(ascending=False).index
sns.boxplot(data=city_df, x='City', y='AQI', order=order, palette='crest')
plt.title('AQI Distribution Across Top 10 Monitored Indian Cities', fontsize=13, fontweight='bold', pad=10)
plt.ylabel('AQI Value', fontsize=11)
plt.xlabel('City (Ranked by Median AQI)', fontsize=11)
plt.axhline(50, color='#10B981', linestyle=':', label='Good Threshold (50)')
plt.axhline(100, color='#84CC16', linestyle=':', label='Satisfactory Threshold (100)')
plt.axhline(200, color='#F59E0B', linestyle='--', label='Moderate Threshold (200)')
plt.axhline(300, color='#EF4444', linestyle='--', label='Poor Threshold (300)')
plt.legend(loc='upper right')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


### Interpretation:
- Landlocked northern cities (e.g. Delhi, Patna, Lucknow) experience markedly higher median AQI and severe upper outlier spikes compared to coastal cities (e.g. Mumbai, Chennai) or southern tablelands (Bengaluru), which benefit from sea breezes and better aerosol dispersion.


## 8. Summary of Findings & Pipeline Preparation
1. **Feature Selection**: The 12 chemical pollutants PM2.5, PM10, NO, NO2, NOx, NH3, CO, SO2, O3, Benzene, Toluene, Xylene form the feature set.
2. **Target Leakage Prevention**: We strictly remove AQI (the numerical target) and Date/City identifiers so the classifier learns pure environmental chemistry patterns.
3. **Missing Value Strategy**: Imputation with median pollutant values inside a scikit-learn pipeline ensures no lookahead data leakage between train and test splits.
4. **Model Choice**: Because the relationship between non-linear CPCB breakpoints and multi-pollutant maximums is piece-wise non-linear, ensemble tree models (Random Forest) are theoretically positioned to outperform linear baselines.
